# ARC-AGI-3 arcade policy evaluator

This notebook is the visual front-end for the same evaluator used by `scripts/evaluate_arcade.py`. The environment loop and official scorecard are fixed; the policy is a replaceable compartment.

## 1. One-time setup

From WSL, run:

```bash
make setup
make arcade-cache
```

`make arcade-cache` discovers and downloads the currently public ARC games. Afterward this notebook uses the cached arcade in offline mode.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from arc_fun.arcade_eval import evaluate_game, list_games
from arc_fun.policy_registry import available_policies

## 2. See the local arcade

This is an offline check of what is actually cached on your machine.

In [ ]:
games = pd.DataFrame(list_games("offline"))
print(f"Cached games: {len(games)}")
games

## 3. Choose a game and policy

Only this configuration block needs to change for ordinary experiments. `null` is the current deterministic baseline.

In [ ]:
GAME_ID = "ls20"
POLICY_NAME = "null"
SEED = 0
MAX_ACTIONS = 80
RENDER_MODE = None  # use "terminal-fast" in a terminal-oriented run

print("Policies:", available_policies())

## 4. Run a real local evaluation

The call below creates the official cached ARC environment, runs the selected policy through real `env.step(...)` interactions, and reads the official local scorecard.

In [ ]:
result = evaluate_game(
    GAME_ID,
    policy_name=POLICY_NAME,
    seed=SEED,
    max_actions=MAX_ACTIONS,
    mode="offline",
    render_mode=RENDER_MODE,
)

pd.DataFrame([{
    "game_id": result.game_id,
    "policy": result.policy,
    "seed": result.seed,
    "final_state": result.final_state,
    "levels_completed": result.levels_completed,
    "actions_taken": result.actions_taken,
    "policy_seconds": result.policy_seconds,
    "official_local_score": result.official_local_score,
}])

In [ ]:
result.scorecard

## 5. Replace the policy, not the evaluator

Future policies belong in the `arc_fun` package and are registered in `policy_registry.py`. The game loader, action loop, and official scorecard should stay unchanged. That lets us compare a hand-written policy, a grammar-generated program, or a world-model + MCTS policy through the same objective evaluation path.